In [1]:
import sys
path='/home/tomas/Ulmer-Berechnung/alps2qutipplus-april/alps2qutipplus-main/'

sys.path.insert(1, path) 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qutip
import scipy.linalg as linalg

from alpsqutip import (build_system, list_models_in_alps_xml,
                       list_geometries_in_alps_xml, graph_from_alps_xml,
                       model_from_alps_xml,
                       restricted_maxent_toolkit as me)

from alpsqutip.operators.states.utils import safe_exp_and_normalize ## function used to safely and robustly map K-states to states
from alpsqutip.operators.states.meanfield import (one_body_from_qutip_operator, 
                                                  project_product_operator_as_n_body_operator,
                                                  
                                                  project_operator_to_m_body, 
                                                  project_to_n_body_operator,
                                                  project_qutip_operator_to_m_body,
                                                  project_meanfield,
                                                  self_consistent_project_meanfield,
                                                  self_consistent_quadratic_mfa)

from alpsqutip.operators import (
    ScalarOperator,
    LocalOperator,
    OneBodyOperator,
    Operator,
    ProductOperator,
    ScalarOperator,
    SumOperator,
    QutipOperator
)

from alpsqutip.scalarprod import gram_matrix
from alpsqutip.operators.states.gibbs import GibbsDensityOperator

In [10]:
params={}

params['size']=2
params['Jx']=1.; params['Jy'] = .75*params['Jx']; params['Jz']=1.05*params['Jx']

from scipy.optimize import root, fsolve
Ffactor=np.real(max(np.roots(np.poly1d([1, 0, -(params['Jx']*params['Jy']+params['Jx']*params['Jy']+params['Jy']*params['Jz']), 
                           -2*params['Jx']*params['Jy']*params['Jz']]))))
chi_y=fsolve(lambda x,y: x*np.arcsinh(x)-np.sqrt(x**2+1)-y, 1e-1, args=(0))[0]
vLR=4*Ffactor*chi_y

In [11]:
system=build_system(geometry_name= "open chain lattice",model_name="spin", 
                    L=params['size'], J=1)

sites=[s for s in system.sites]
sx_ops=[system.site_operator("Sx", '1[' + str(a) + ']') for a in range(len(system.sites))]
sy_ops=[system.site_operator("Sy", '1[' + str(a) + ']') for a in range(len(system.sites))]
sz_ops=[system.site_operator("Sz", '1[' + str(a) + ']') for a in range(len(system.sites))]

Sz = system.global_operator("Sz")
H = (
    params['Jx'] * sum(sx_ops[i] * sx_ops[i+1] for i in range(len(sites)-1)) +
    params['Jy'] * sum(sy_ops[i] * sy_ops[i+1] for i in range(len(sites)-1)) +
    params['Jz'] * sum(sz_ops[i] * sz_ops[i+1] for i in range(len(sites)-1))
)

idop = [system.site_operator('identity@1[' + str(a) + ']') for a in range(len(system.sites))]
idop = idop[0]*idop[1]#*idop[2]#*idop[3]#*idop[4]*idop[5]*idop[6]

loading model spin  over graph open chain lattice


In [12]:
HBB0=[idop, 
      system.site_operator('Sx', '1[0]'), 
      system.site_operator('Sy', '1[0]'), 
      system.site_operator('Sz', '1[0]')]


phi0 = np.array([.0, -.25, -.25, 
                 100])

K0 = me.k_state_from_phi_basis(phi0, HBB0)
sigma0 = GibbsDensityOperator(-K0)
phi0[0] = np.log(sigma0.tr())
K0 = me.k_state_from_phi_basis(phi0, HBB0)
sigma0 = GibbsDensityOperator(-K0)

[(sigma0 * op).tr() for op in sz_ops] 

[-0.4999968750292965, 0.0]

In [8]:
## exact dynamics
exact_K_at_timet=qutip.mesolve(H=H.to_qutip(), rho0=K0.to_qutip(), tlist=timespan)
exact_rho_at_timet=[safe_exp_and_normalize(K_at_timeti)[0] for K_at_timeti in exact_K_at_timet.states]

evs_obs_ex=[qutip.expect(rho_at_timeti.ptrace(ptrace_list), obs_SzA) for rho_at_timeti in exact_rho_at_timet]

NameError: name 'timespan' is not defined

In [46]:
Ktgt=K0 + 1j*me.commutator(H, K0) * .1 + 1j*me.commutator(H, 1j*me.commutator(H, K0) * .1) * .1
Ktgt+= 1j*me.commutator(H, 1j*me.commutator(H, 1j*me.commutator(H, K0) * .1) * .1) * .1

Ktgt = Ktgt.simplify()

In [47]:
Ktgt.to_qutip_operator()

qutip interface operator for 1 x  
Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[ 4.99921875e+01+0.00000000e+00j -2.95898438e-04+1.94384766e-04j
  -5.36671875e-03-5.56312500e-03j  0.00000000e+00-3.51562500e-03j
   1.30961551e-01-1.17839572e-01j  0.00000000e+00+0.00000000e+00j
   3.28125000e-02-6.24351563e-01j  8.20312500e-05-8.20312500e-05j]
 [-2.95898438e-04-1.94384766e-04j  4.99921875e+01+0.00000000e+00j
   0.00000000e+00-2.46093750e-02j -5.96226562e-03-4.98439453e-03j
   1.87500000e-01-9.84375000e-03j  1.30936941e-01-1.17864182e-01j
   5.74218750e-04-5.74218750e-04j -3.28125000e-02-6.24351563e-01j]
 [-5.36671875e-03+5.56312500e-03j  0.00000000e+00+2.46093750e-02j
   4.96171875e+01+0.00000000e+00j  2.71289063e-04-2.18994141e-04j
  -2.29687500e-01-4.32124219e+00j -5.74218750e-04+5.74218750e-04j
   1.18022434e-01-1.30821756e-01j  0.00000000e+00+0.00000000e+00j]
 [ 0.00000000e+00+3.51562500e-03j -5.96226562e-03+4.98439453e-03j
  

In [49]:
project_operator_to_m_body(full_operator=Ktgt, 
                           m_max=2, 
                           sigma_0=sigma0).to_qutip_operator()

qutip interface operator for 1 x  
Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[ 4.99921875e+01+0.00000000e+00j -6.15234369e-05-3.99902343e-05j
  -5.61281250e-03-5.23500000e-03j  0.00000000e+00-3.51562500e-03j
   1.30961551e-01-1.17839572e-01j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00-6.24351562e-01j  0.00000000e+00+0.00000000e+00j]
 [-6.15234369e-05+3.99902343e-05j  4.99921875e+01+0.00000000e+00j
   0.00000000e+00-2.46093750e-02j -5.71617188e-03-5.31251953e-03j
   0.00000000e+00-9.84375000e-03j  1.30936941e-01-1.17864182e-01j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00-6.24351562e-01j]
 [-5.61281250e-03+5.23500000e-03j  0.00000000e+00+2.46093750e-02j
   4.96171875e+01+0.00000000e+00j  3.69140621e-05+1.53808593e-05j
   0.00000000e+00-4.32124219e+00j  0.00000000e+00+0.00000000e+00j
   1.18022434e-01-1.30821756e-01j  0.00000000e+00+0.00000000e+00j]
 [ 0.00000000e+00+3.51562500e-03j -5.71617188e-03+5.31251953e-03j
  

In [13]:
HBB0=[idop, 
      system.site_operator('Sz', '1[0]'), 
      system.site_operator('Sz', '1[1]')]


phi0 = np.array([.1, -1.25, -1.25])

K0 = me.k_state_from_phi_basis(phi0, HBB0)
sigma0 = GibbsDensityOperator(-K0)
phi0[0] = np.log(sigma0.tr())
K0 = me.k_state_from_phi_basis(phi0, HBB0)
sigma0 = GibbsDensityOperator(-K0)

[(sigma0 * op).tr() for op in sz_ops] 

[0.27729986117469113, 0.2772998611746912]

In [14]:
op=system.site_operator("Sz@1[0]")*system.site_operator("Sz@1[1]")

sigma0 ## estado separable e**(a*Sz1 + b*Sz2)

In [15]:
sz = qutip.sigmaz()
id2 = qutip.qeye(2)

Sz1 = .5*qutip.tensor(sz, id2); Sz2 = .5*qutip.tensor(id2, sz)
Sz1Sz2 = Sz1 * Sz2
exp_Sz1 = qutip.expect(Sz1, sigma0.to_qutip())
exp_Sz2 = qutip.expect(Sz2, sigma0.to_qutip())

Sz1Sz2_mf = (Sz1 - exp_Sz1) * exp_Sz2 + (Sz2 - exp_Sz2) * exp_Sz1 + exp_Sz1*exp_Sz2

In [16]:
sz = qutip.sigmaz()
id2 = qutip.qeye(2)

Sz1 = .5*qutip.tensor(sz, id2); Sz2 = .5*qutip.tensor(id2, sz)
Sz1Sz2 = Sz1 * Sz2
exp_Sz1 = qutip.expect(Sz1, sigma0.to_qutip())
exp_Sz2 = qutip.expect(Sz2, sigma0.to_qutip())

Sz1Sz2_mf = (Sz1 - exp_Sz1) * exp_Sz2 + (Sz2 - exp_Sz2) * exp_Sz1 + exp_Sz1*exp_Sz2

In [17]:
Sz1Sz2_mf

Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[-0.35419507  0.          0.          0.        ]
 [ 0.         -0.07689521  0.          0.        ]
 [ 0.          0.         -0.07689521  0.        ]
 [ 0.          0.          0.          0.20040465]]

In [18]:
project_operator_to_m_body(full_operator=system.site_operator("Sz@1[0]")*system.site_operator("Sz@1[1]"), 
                           m_max=1, 
                           sigma_0=sigma0).to_qutip_operator()

qutip interface operator for 1 x  
Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[-0.35419507  0.          0.          0.        ]
 [ 0.         -0.07689521  0.          0.        ]
 [ 0.          0.         -0.07689521  0.        ]
 [ 0.          0.          0.          0.20040465]]

In [18]:
project_product_operator_as_n_body_operator(operator=system.site_operator("Sz@1[0]")*system.site_operator("Sz@1[1]"),
                                            nmax=1,
                                            sigma=sigma0).to_qutip()

Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[0.05572475 0.         0.         0.        ]
 [0.         0.06163762 0.         0.        ]
 [0.         0.         0.07704778 0.        ]
 [0.         0.         0.         0.08296065]]

In [ ]:
( Sz1-<Sz1>) <Sz2>+ <Sz1>(Sz2 -<Sz2>) +  <Sz1><Sz2>

In [85]:
project_product_operator_as_n_body_operator(operator=op,
                                            nmax=1,
                                            sigma=sigma0).to_qutip()

Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[-0.35419507  0.          0.          0.        ]
 [ 0.         -0.07689521  0.          0.        ]
 [ 0.          0.         -0.07689521  0.        ]
 [ 0.          0.          0.          0.20040465]]

In [84]:
project_product_operator_as_n_body_operator(operator=op,
                                            nmax=1,
                                            sigma=sigma0)

(
  0.076895213007503 * (
  
   )
  +  0.2772998611746912 * (
  ([[-0.77729986+0.j  0.        +0.j]
 [ 0.        +0.j  0.22270014+0.j]] <-  1[0])
   )
  +  0.2772998611746912 * (
  ([[-0.77729986+0.j  0.        +0.j]
 [ 0.        +0.j  0.22270014+0.j]] <-  1[1])
   )
)

In [73]:
(op*sigma0).tr()

0.076895213007503

In [61]:
opq=.25*qutip.tensor(qutip.sigmaz(), qutip.sigmaz())

qutip.tensor(qutip.sigmaz(), qutip.qeye(2))

Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[ 1.  0.  0.  0.]
 [ 0.  1.  0.  0.]
 [ 0.  0. -1.  0.]
 [ 0.  0.  0. -1.]]

In [78]:
from qutip import *
import numpy as np

# Define Pauli Z operators on 2 sites
sz = sigmaz()
id2 = qeye(2)

# Define S^z_1 and S^z_2
Sz1 = tensor(sz, id2)
Sz2 = tensor(id2, sz)

# Define the 2-body operator S^z_1 S^z_2
Sz1Sz2 = Sz1 * Sz2

# Define a product state rho = rho1 ⊗ rho2
# For example: |+><+| ⊗ |0><0|

# Compute the expectations
exp_Sz1 = expect(Sz1, sigma0.to_qutip())
exp_Sz2 = expect(Sz2, sigma0.to_qutip())

# Construct the mean-field approximation
Sz1_mf = exp_Sz2 * Sz1
Sz2_mf = exp_Sz1 * Sz2
const_term = exp_Sz1 * exp_Sz2 * tensor(id2, id2)

Sz1Sz2_mf = Sz1_mf + Sz2_mf - const_term

# Optional: Compare original and mean-field
print("Exact expectation value ⟨Sz1Sz2⟩ =", expect(Sz1Sz2, rho))
print("Mean-field approximation ⟨Sz1Sz2_mf⟩ =", expect(Sz1Sz2_mf, rho))


Exact expectation value ⟨Sz1Sz2⟩ = 0.0
Mean-field approximation ⟨Sz1Sz2_mf⟩ = -0.862180574379394


In [79]:
Sz1Sz2

Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[ 1.  0.  0.  0.]
 [ 0. -1.  0.  0.]
 [ 0.  0. -1.  0.]
 [ 0.  0.  0.  1.]]

In [80]:
Sz1Sz2_mf

Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[-1.4167803   0.          0.          0.        ]
 [ 0.         -0.30758085  0.          0.        ]
 [ 0.          0.         -0.30758085  0.        ]
 [ 0.          0.          0.          0.80161859]]

In [41]:
(op.to_qutip().ptrace([0])) * ((op.to_qutip().ptrace([1])) * sigma0.to_qutip().ptrace([1])).tr() 

Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[0. 0.]
 [0. 0.]]